In [2]:
import pandas as pd
import numpy as np
import yaml

# Load config so paths stay consistent with the pipeline
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

PATHS = {
    'diabetes_130us': config['datasets']['diabetes_130us']['raw_file'],
    'home_credit':    config['datasets']['home_credit']['raw_file'],
    'acs_income':     config['datasets']['acs_income']['raw_file'],
}

def section(title):
    print('\n' + '='*70)
    print(f'  {title}')
    print('='*70)

In [3]:
def run_eda(name, path, target_col, drop_cols=None, missing_markers=None):
    drop_cols = drop_cols or []
    missing_markers = missing_markers or []

    section(f'DATASET: {name}')
    df = pd.read_csv(path)

    # Replace dataset-specific missing markers with NaN
    for marker in missing_markers:
        df.replace(marker, np.nan, inplace=True)

    print(f'\n--- Shape')
    print(f'  Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')

    print(f'\n--- Column dtypes')
    print(df.dtypes.to_string())

    print(f'\n--- Target column: "{target_col}"')
    print(df[target_col].value_counts(dropna=False))

    print(f'\n--- Missing values (columns with any nulls)')
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print('  None')
    else:
        missing_pct = (missing / len(df) * 100).round(2)
        print(pd.DataFrame({'count': missing, 'pct': missing_pct}).to_string())

    print(f'\n--- Numeric columns — summary stats')
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in num_cols:
        num_cols.remove(target_col)
    for c in drop_cols:
        if c in num_cols:
            num_cols.remove(c)
    if num_cols:
        print(df[num_cols].describe().round(3).to_string())
    else:
        print('  No numeric columns (excluding target/drop)')

    print(f'\n--- Categorical columns — cardinality')
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    for c in drop_cols:
        if c in cat_cols:
            cat_cols.remove(c)
    if cat_cols:
        card = pd.DataFrame({
            'unique': [df[c].nunique() for c in cat_cols],
            'top_value': [df[c].mode()[0] if not df[c].mode().empty else 'N/A' for c in cat_cols],
            'top_freq_pct': [(df[c].value_counts(normalize=True).iloc[0]*100).round(2) if df[c].notna().any() else 0 for c in cat_cols],
        }, index=cat_cols)
        print(card.to_string())
    else:
        print('  No categorical columns')

    print(f'\n--- High-cardinality categoricals (> 20 unique values)')
    if cat_cols:
        high_card = [c for c in cat_cols if df[c].nunique() > 20]
        print(f'  {high_card if high_card else "None"}')
    else:
        print('  N/A')

    print(f'\n--- Duplicate rows')
    dupes = df.duplicated().sum()
    print(f'  {dupes:,} ({dupes/len(df)*100:.2f}%)')

    print(f'\n--- Columns to drop (from config)')
    print(f'  {drop_cols}')

    print(f'\n--- First 3 rows (raw)')
    print(df.head(3).to_string())

    return df

---
## 1 — Diabetes 130-US

In [4]:
df_diabetes = run_eda(
    name='diabetes_130us',
    path='../data/raw/diabetes_130us.csv',
    target_col='readmitted',
    drop_cols=['encounter_id', 'patient_nbr'],
    missing_markers=['?'],
)


  DATASET: diabetes_130us

--- Shape
  Rows: 101,766   Columns: 50

--- Column dtypes
encounter_id                 int64
patient_nbr                  int64
race                        object
gender                      object
age                         object
weight                      object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
payer_code                  object
medical_specialty           object
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nat

---
## 2 — Home Credit Default Risk

In [5]:
df_home = run_eda(
    name='home_credit',
    path='../data/raw/home_credit_default_risk.csv',
    target_col='TARGET',
    drop_cols=['SK_ID_CURR'],
    missing_markers=[],
)


  DATASET: home_credit

--- Shape
  Rows: 307,511   Columns: 122

--- Column dtypes
SK_ID_CURR                        int64
TARGET                            int64
NAME_CONTRACT_TYPE               object
CODE_GENDER                      object
FLAG_OWN_CAR                     object
FLAG_OWN_REALTY                  object
CNT_CHILDREN                      int64
AMT_INCOME_TOTAL                float64
AMT_CREDIT                      float64
AMT_ANNUITY                     float64
AMT_GOODS_PRICE                 float64
NAME_TYPE_SUITE                  object
NAME_INCOME_TYPE                 object
NAME_EDUCATION_TYPE              object
NAME_FAMILY_STATUS               object
NAME_HOUSING_TYPE                object
REGION_POPULATION_RELATIVE      float64
DAYS_BIRTH                        int64
DAYS_EMPLOYED                     int64
DAYS_REGISTRATION               float64
DAYS_ID_PUBLISH                   int64
OWN_CAR_AGE                     float64
FLAG_MOBIL                        i

---
## 3 — ACS Income

In [6]:
df_acs = run_eda(
    name='acs_income',
    path='../data/raw/acs_income_ca_2018.csv',
    target_col='PINCP',
    drop_cols=[],
    missing_markers=[],
)


  DATASET: acs_income

--- Shape
  Rows: 195,665   Columns: 11

--- Column dtypes
AGEP     float64
COW      float64
SCHL     float64
MAR      float64
OCCP     float64
POBP     float64
RELP     float64
WKHP     float64
SEX      float64
RAC1P    float64
PINCP       bool

--- Target column: "PINCP"
PINCP
False    115330
True      80335
Name: count, dtype: int64

--- Missing values (columns with any nulls)
  None

--- Numeric columns — summary stats
             AGEP         COW        SCHL         MAR        OCCP        POBP        RELP        WKHP         SEX       RAC1P
count  195665.000  195665.000  195665.000  195665.000  195665.000  195665.000  195665.000  195665.000  195665.000  195665.000
mean       42.735       2.144      18.470       2.654    4020.583      94.396       2.510      37.867       1.472       3.073
std        14.885       1.888       3.942       1.846    2637.180     123.507       4.445      13.019       0.499       2.916
min        17.000       1.000       1.000    

In [7]:
for name, df, target in [
    ('diabetes_130us', df_diabetes, 'readmitted'),
    ('home_credit',    df_home,     'TARGET'),
    ('acs_income',     df_acs,      'PINCP'),
]:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
    missing_cols = df.isnull().sum()
    missing_cols = missing_cols[missing_cols > 0]

    print(f'\n[{name}]')
    print(f'  shape          : {df.shape}')
    print(f'  numeric cols   : {len(num_cols)}')
    print(f'  categorical cols: {len(cat_cols)}')
    print(f'  cols with nulls: {len(missing_cols)}')
    print(f'  target         : {target} — {df[target].value_counts(normalize=True).round(3).to_dict()}')
    print(f'  high-card cats : {[c for c in cat_cols if df[c].nunique() > 20]}')
    print(f'  all columns    : {df.columns.tolist()}')


[diabetes_130us]
  shape          : (101766, 50)
  numeric cols   : 13
  categorical cols: 37
  cols with nulls: 9
  target         : readmitted — {'NO': 0.539, '>30': 0.349, '<30': 0.112}
  high-card cats : ['medical_specialty', 'diag_1', 'diag_2', 'diag_3']
  all columns    : ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitaz

In [10]:
# Null percentages for all null columns
print(df_diabetes.isnull().sum()[df_diabetes.isnull().sum() > 0] / len(df_diabetes) * 100)

# Unique values for drug columns
drug_cols = ['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
             'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
             'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
             'examide','citoglipton','insulin','glyburide-metformin','glipizide-metformin',
             'glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone']
for c in drug_cols:
    print(f"{c}: {df_diabetes[c].unique()}")

# diag columns — unique count and sample values
for c in ['diag_1','diag_2','diag_3']:
    print(f"{c}: {df_diabetes[c].nunique()} unique — sample: {df_diabetes[c].dropna().head(10).tolist()}")

# medical_specialty — full value counts
print(df_diabetes['medical_specialty'].value_counts(dropna=False))

race                  2.233555
weight               96.858479
payer_code           39.557416
medical_specialty    49.082208
diag_1                0.020636
diag_2                0.351787
diag_3                1.398306
max_glu_serum        94.746772
A1Cresult            83.277322
dtype: float64
metformin: ['No' 'Steady' 'Up' 'Down']
repaglinide: ['No' 'Up' 'Steady' 'Down']
nateglinide: ['No' 'Steady' 'Down' 'Up']
chlorpropamide: ['No' 'Steady' 'Down' 'Up']
glimepiride: ['No' 'Steady' 'Down' 'Up']
acetohexamide: ['No' 'Steady']
glipizide: ['No' 'Steady' 'Up' 'Down']
glyburide: ['No' 'Steady' 'Up' 'Down']
tolbutamide: ['No' 'Steady']
pioglitazone: ['No' 'Steady' 'Up' 'Down']
rosiglitazone: ['No' 'Steady' 'Up' 'Down']
acarbose: ['No' 'Steady' 'Up' 'Down']
miglitol: ['No' 'Steady' 'Down' 'Up']
troglitazone: ['No' 'Steady']
tolazamide: ['No' 'Steady' 'Up']
examide: ['No']
citoglipton: ['No']
insulin: ['No' 'Up' 'Steady' 'Down']
glyburide-metformin: ['No' 'Steady' 'Down' 'Up']
glipizide-metfor

In [8]:
# Null percentage per column, sorted
null_pct = df_home.isnull().sum() / len(df_home) * 100
print(null_pct[null_pct > 0].sort_values(ascending=False).to_string())

# Check if AVG/MODE/MEDI nulls are correlated (same rows missing)
avg_cols = [c for c in df_home.columns if c.endswith('_AVG')]
mode_cols = [c for c in df_home.columns if c.endswith('_MODE')]
medi_cols = [c for c in df_home.columns if c.endswith('_MEDI')]
print("AVG null rows:", df_home[avg_cols].isnull().all(axis=1).sum())
print("MODE null rows:", df_home[mode_cols].isnull().all(axis=1).sum())
print("MEDI null rows:", df_home[medi_cols].isnull().all(axis=1).sum())
print("Total rows:", len(df_home))

# DAYS_EMPLOYED anomaly
print(df_home['DAYS_EMPLOYED'].describe())
print("Rows with DAYS_EMPLOYED == 365243:", (df_home['DAYS_EMPLOYED'] == 365243).sum())

# AMT_INCOME_TOTAL and AMT_CREDIT distributions
print(df_home[['AMT_INCOME_TOTAL','AMT_CREDIT']].describe())

COMMONAREA_MEDI                 69.872297
COMMONAREA_AVG                  69.872297
COMMONAREA_MODE                 69.872297
NONLIVINGAPARTMENTS_MEDI        69.432963
NONLIVINGAPARTMENTS_MODE        69.432963
NONLIVINGAPARTMENTS_AVG         69.432963
FONDKAPREMONT_MODE              68.386172
LIVINGAPARTMENTS_MODE           68.354953
LIVINGAPARTMENTS_MEDI           68.354953
LIVINGAPARTMENTS_AVG            68.354953
FLOORSMIN_MODE                  67.848630
FLOORSMIN_MEDI                  67.848630
FLOORSMIN_AVG                   67.848630
YEARS_BUILD_MODE                66.497784
YEARS_BUILD_MEDI                66.497784
YEARS_BUILD_AVG                 66.497784
OWN_CAR_AGE                     65.990810
LANDAREA_AVG                    59.376738
LANDAREA_MEDI                   59.376738
LANDAREA_MODE                   59.376738
BASEMENTAREA_MEDI               58.515956
BASEMENTAREA_AVG                58.515956
BASEMENTAREA_MODE               58.515956
EXT_SOURCE_1                    56

In [9]:
# Unique values for all census code columns
cat_cols = ['COW','SCHL','MAR','OCCP','POBP','RELP','SEX','RAC1P']
for c in cat_cols:
    vals = sorted(df_acs[c].unique().tolist())
    print(f"{c} ({len(vals)} unique): {vals}")

# Distribution of continuous columns
print(df_acs[['AGEP','WKHP']].describe())

COW (8 unique): [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
SCHL (24 unique): [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0]
MAR (5 unique): [1.0, 2.0, 3.0, 4.0, 5.0]
OCCP (529 unique): [10.0, 20.0, 40.0, 51.0, 52.0, 60.0, 101.0, 102.0, 110.0, 120.0, 135.0, 136.0, 137.0, 140.0, 150.0, 160.0, 205.0, 220.0, 230.0, 300.0, 310.0, 335.0, 340.0, 350.0, 360.0, 410.0, 420.0, 425.0, 440.0, 500.0, 510.0, 520.0, 530.0, 540.0, 565.0, 600.0, 630.0, 640.0, 650.0, 700.0, 705.0, 710.0, 725.0, 726.0, 735.0, 750.0, 800.0, 810.0, 820.0, 830.0, 845.0, 850.0, 860.0, 900.0, 910.0, 930.0, 940.0, 960.0, 1005.0, 1006.0, 1007.0, 1010.0, 1021.0, 1022.0, 1031.0, 1032.0, 1050.0, 1065.0, 1105.0, 1106.0, 1108.0, 1200.0, 1220.0, 1240.0, 1305.0, 1306.0, 1310.0, 1320.0, 1340.0, 1350.0, 1360.0, 1400.0, 1410.0, 1420.0, 1430.0, 1440.0, 1450.0, 1460.0, 1520.0, 1530.0, 1541.0, 1545.0, 1551.0, 1555.0, 1560.0, 1600.0, 1610.0, 1640.0, 1650.0,